In [1]:
import pandas as pd

In [2]:
OUTPUT_DIR = r"C:\Users\radet\Praksa\automatisation\output"

In [3]:
import os
import sys

In [4]:
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [5]:
from utils.metrics import has_invalid_extra
from utils.load import load_data, optimize_dataframe, load_event_data

In [6]:
import pyarrow as pa
import pyarrow.parquet as pq

In [31]:
print("Loading data in chunks...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

writers = {}

total_before = 0
total_after = 0

for i, chunk in enumerate(load_data(chunksize=10_000)):
    chunk = optimize_dataframe(chunk)

    total_before += len(chunk)

    mask = chunk["extra"].map(has_invalid_extra)
    chunk = chunk.loc[~mask].copy()

    if chunk.empty:
        del chunk, mask
        continue

    total_after += len(chunk)

    chunk["insertedTS"] = pd.to_datetime(chunk["insertedTS"])
    chunk["event_date"] = chunk["insertedTS"].dt.date.astype(str)

    for event_date, df_date in chunk.groupby("event_date"):
        file_path = f"{OUTPUT_DIR}/iptv_{event_date}.parquet"

        df_date = df_date.drop(columns=["event_date"])

        table = pa.Table.from_pandas(
            df_date,
            preserve_index=False,
        )

        if event_date not in writers:
            writers[event_date] = pq.ParquetWriter(
                file_path,
                table.schema,
            )

        writers[event_date].write_table(table)

        print(
            f"Chunk {i}, date={event_date}: rows={len(df_date)}"
        )

        del df_date, table

    del chunk, mask

for writer in writers.values():
    writer.close()

if not writers:
    raise ValueError("No valid rows after cleaning")

print(f"Rows before cleaning: {total_before}")
print(f"Rows after cleaning: {total_after}")
print(f"Saved parquet files to: {OUTPUT_DIR}")


Loading data in chunks...


Chunk 0, date=2026-05-20: rows=9911
Chunk 1, date=2026-05-20: rows=9915
Chunk 2, date=2026-05-20: rows=9920
Chunk 3, date=2026-05-20: rows=9919
Chunk 4, date=2026-05-20: rows=9919
Chunk 5, date=2026-05-20: rows=9916
Chunk 6, date=2026-05-20: rows=9918
Chunk 7, date=2026-05-20: rows=9923
Chunk 8, date=2026-05-20: rows=9926
Chunk 9, date=2026-05-20: rows=9925
Chunk 10, date=2026-05-20: rows=9919
Chunk 11, date=2026-05-20: rows=9918
Chunk 12, date=2026-05-20: rows=9915
Chunk 13, date=2026-05-20: rows=9913
Chunk 14, date=2026-05-20: rows=9896
Chunk 15, date=2026-05-20: rows=9906
Chunk 16, date=2026-05-20: rows=9908
Chunk 17, date=2026-05-20: rows=9901
Chunk 18, date=2026-05-20: rows=9911
Chunk 19, date=2026-05-20: rows=9918
Chunk 20, date=2026-05-20: rows=9909
Chunk 21, date=2026-05-20: rows=9934
Chunk 22, date=2026-05-20: rows=9950
Chunk 23, date=2026-05-20: rows=9958
Chunk 24, date=2026-05-20: rows=9948
Chunk 25, date=2026-05-20: rows=9949
Chunk 26, date=2026-05-20: rows=9945
Chunk 27, d

In [42]:
df = pd.read_parquet(r"C:\Users\radet\Praksa\automatisation\output\iptv_2026-04-21.parquet")

In [44]:
df_m = df[(df["type"] == "RESTARTUsage") & (df["name"] == "Divlje p─ìele")]
df_m

,devRef,name,type,duration,extra,insertedTS,timeZone
268,9294460022083742,Divlje p─ìele,RESTARTUsage,300052,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 00:00:00,UTC+02:00
921,9905460028863251,Divlje p─ìele,RESTARTUsage,263164,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 00:00:00,UTC+01:00
1653,9881740028396502,Divlje p─ìele,RESTARTUsage,300060,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 00:00:07,UTC+01:00
5045,2317940024581191,Divlje p─ìele,RESTARTUsage,300069,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 00:00:26,UTC+01:00
5876,3259170026849221,Divlje p─ìele,RESTARTUsage,300152,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 00:00:26,UTC+02:00
...,...,...,...,...,...,...,...
13405130,2317940024581191,Divlje p─ìele,RESTARTUsage,300124,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 23:59:28,UTC+01:00
13405441,4757870011694802,Divlje p─ìele,RESTARTUsage,300037,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 23:59:38,UTC+02:00
13407950,59835200098426700001,Divlje p─ìele,RESTARTUsage,300067,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 23:59:51,UTC+02:00
13408332,5031300025507981,Divlje p─ìele,RESTARTUsage,300059,"{""title"":""Divlje p─ìele"",""genre"":""Film"",""start...",2026-04-21 23:59:54,UTC+02:00


In [7]:
df = load_event_data("RESTARTUsage")

In [8]:
df

,devRef,name,duration,extra,insertedTS,timeZone
0,3276800023118281,Divlje pčele,300017,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 00:00:00,UTC+02:00
1,77083000317985700001,Divlje pčele,300053,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 00:00:00,UTC+02:00
2,702700014128941,Divlje pčele,300046,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 00:00:06,UTC+02:00
3,9450000193616600001,Divlje pčele,300050,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 00:00:09,UTC+02:00
4,5291990029961411,Divlje pčele,300075,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 00:00:09,UTC+02:00
...,...,...,...,...,...,...
17283,2921160032869341,Divlje pčele,300038,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 23:59:50,UTC+02:00
17284,10038160030369902,Divlje pčele,300068,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 23:59:51,UTC+02:00
17285,10024510030143041,Divlje pčele,300183,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 23:59:51,UTC+02:00
17286,1328420019076531,Divlje pčele,300059,"{""title"":""Divlje pčele"",""genre"":""Film"",""startT...",2026-05-20 23:59:56,UTC+02:00
